# 02 - Curve Bootstrap and Forward Rates
Load sample Treasury inputs, bootstrap a spot curve, derive forward rates, and visualize both.

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd().resolve().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from fixed_income_toolkit.io import read_treasuries_csv
from fixed_income_toolkit.curve import bootstrap_spot_curve, derive_forward_curve

treasury_path = repo_root / "data" / "sample" / "treasuries.csv"

In [ ]:
instruments = read_treasuries_csv(treasury_path)
spot_points = bootstrap_spot_curve(instruments)
fwd_points = derive_forward_curve(spot_points)

spot_df = pd.DataFrame({
    "tenor_years": [p.tenor_years for p in spot_points],
    "zero_rate": [p.zero_rate for p in spot_points],
    "discount_factor": [p.discount_factor for p in spot_points],
})

fwd_df = pd.DataFrame({
    "start_tenor": [p.start_tenor for p in fwd_points],
    "end_tenor": [p.end_tenor for p in fwd_points],
    "forward_rate": [p.forward_rate for p in fwd_points],
})

spot_df, fwd_df.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(spot_df["tenor_years"], spot_df["zero_rate"] * 100, marker="o")
ax[0].set_title("Spot (Zero) Curve")
ax[0].set_xlabel("Tenor (years)")
ax[0].set_ylabel("Zero Rate (%)")
ax[0].grid(alpha=0.3)

if not fwd_df.empty:
    fwd_x = [f"{int(s)}->{int(e)}" for s, e in zip(fwd_df["start_tenor"], fwd_df["end_tenor"])]
    ax[1].plot(fwd_x, fwd_df["forward_rate"] * 100, marker="o", color="tab:orange")
ax[1].set_title("Forward Rates by Segment")
ax[1].set_xlabel("Segment")
ax[1].set_ylabel("Forward Rate (%)")
ax[1].grid(alpha=0.3)
plt.tight_layout()

In [ ]:
outputs_dir = repo_root / "outputs"
outputs_dir.mkdir(exist_ok=True)
spot_df.to_csv(outputs_dir / "spot_curve_from_notebook.csv", index=False)
fwd_df.to_csv(outputs_dir / "forward_curve_from_notebook.csv", index=False)
outputs_dir